In [2]:
import cv2
import numpy as np

In [3]:
from ultralytics import YOLO

# Load model pretrained (COCO)
model = YOLO("yolov8n.pt")

In [4]:

from sklearn.cluster import KMeans

def detect_traffic_light_color_kmeans(roi, k=4):
    hsv_roi=cv2.cvtColor(roi, cv2.COLOR_BGR2HSV)
    pixels = hsv_roi.reshape(-1, 3)
    pixels = pixels[pixels[:, 2] > 100]  # V > 50

    if len(pixels) == 0:
        return "UNKNOWN"

    kmeans = KMeans(n_clusters=k, n_init=7)
    kmeans.fit(pixels)

    centers = kmeans.cluster_centers_

    centers = np.array(centers)

    scores = centers[:, 1] * centers[:, 2]  # S * V
    best = centers[np.argmax(scores)]

    h, s, v = best

    # 5. điều kiện phân loại màu (Hue)
    if (h < 10) or (h > 170):
        return "RED"
    elif 50 < h < 100:
        return "GREEN"
    elif 15 < h < 35:
        return "YELLOW"
    else:
        return "UNKNOWN"

In [5]:
# cap = cv2.VideoCapture(r"D:\cdio3\Download.mp4")

# while True:
#     ret, frame = cap.read()
#     if not ret:
#         break

#     results = model(frame)

#     for r in results:
#         for box in r.boxes:
#             cls_id = int(box.cls[0])
#             class_name = model.names[cls_id]

#             if class_name == "traffic light":
#                 x1, y1, x2, y2 = map(int, box.xyxy[0])

#                 # cv2.rectangle(frame, (x1, y1), (x2, y2), (0,255,0), 2)
#                 crop = frame[y1:y2, x1:x2]
#                 print(f"crop: {crop.shape}")
#                 color = detect_traffic_light_color_kmeans(crop)

#                 cv2.putText(frame, color, (x1, y1-10),
#                             cv2.FONT_HERSHEY_SIMPLEX, 0.7,
#                             (22,55,67), 2)

#     cv2.imshow("video", frame)

#     if cv2.waitKey(1) == 27:
#         break

# cap.release()
# cv2.destroyAllWindows()

In [6]:
# import cv2

# def get_hsv(event, x, y, flags, param):
#     if event == cv2.EVENT_LBUTTONDOWN:
#         pixel = hsv[y, x]
#         print(f"H: {pixel[0]}, S: {pixel[1]}, V: {pixel[2]}")

# img = cv2.imread(r"D:\cdio3\anhvideo\frame_0293.jpg")
# hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)

# cv2.imshow("image", img)
# cv2.setMouseCallback("image", get_hsv)

# cv2.waitKey(0)
# cv2.destroyAllWindows()

In [ ]:
model = YOLO("yolov8s.pt") 
cap = cv2.VideoCapture(r"D:\cdio3\video_den_do.mp4")
history = {}
count_his = []
count = 0
c_line = 400
color = "green"

while True:
    ret, frame = cap.read()
    if not ret:
        break

    results = model.track(frame, persist=True, tracker="bytetrack.yaml")

    for r in results:
        boxes = r.boxes

        for box in boxes:
            if box.id is None:
                continue

            track_id = int(box.id[0])
            cls = int(box.cls[0])

         
            if cls in [2, 3, 5, 7]:
                x1, y1, x2, y2 = map(int, box.xyxy[0])
                cx = (x1 + x2) // 2
                cy = (y1 + y2) // 2

                if track_id not in history:
                    history[track_id] = cy
                    continue

                prev_cy = history[track_id]
                history[track_id] = cy

              
                if (prev_cy >= c_line) and (cy < c_line) and (track_id not in count_his):
                    if color.lower() == "red":
                        count += 1
                        count_his.append(track_id)
                        print(f"VI PHAM ID: {track_id}")

                
                # cv2.rectangle(frame, (x1,y1), (x2,y2), (0,255,0), 2)
                # cv2.putText(frame, f"ID {track_id}", (x1, y1-10),
                #             cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,255,0), 2)

      
            if cls == 9:
                x1, y1, x2, y2 = map(int, box.xyxy[0])
                crop = frame[y1:y2, x1:x2]

                color = detect_traffic_light_color_kmeans(crop)

                # cv2.putText(frame, color, (x1, y1-10),
                #             cv2.FONT_HERSHEY_SIMPLEX, 0.7,
                #             (0,0,255), 2)

    # vẽ vạch
    # cv2.line(frame, (0, c_line), (frame.shape[1], c_line), (255,0,0), 2)

    cv2.putText(frame, f"Count: {count}", (50,50),
                cv2.FONT_HERSHEY_SIMPLEX, 1, (0,0,255), 2)

    cv2.imshow("Tracking", frame)

    if cv2.waitKey(1) == 27:
        break
cap.release()
cv2.destroyAllWindows()


0: 384x640 9 persons, 3 cars, 4 motorcycles, 4 traffic lights, 1 handbag, 273.1ms
Speed: 4.8ms preprocess, 273.1ms inference, 1.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 9 persons, 3 cars, 4 motorcycles, 4 traffic lights, 1 handbag, 303.5ms
Speed: 61.0ms preprocess, 303.5ms inference, 1.8ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 8 persons, 3 cars, 4 motorcycles, 4 traffic lights, 290.8ms
Speed: 3.7ms preprocess, 290.8ms inference, 1.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 7 persons, 3 cars, 5 motorcycles, 4 traffic lights, 276.5ms
Speed: 5.2ms preprocess, 276.5ms inference, 1.0ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 8 persons, 2 cars, 7 motorcycles, 4 traffic lights, 257.7ms
Speed: 5.1ms preprocess, 257.7ms inference, 1.2ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 8 persons, 2 cars, 6 motorcycles, 3 traffic lights, 247.9ms
Speed: 4.2ms preprocess, 247.9ms inference, 2.2ms postp

In [8]:
# import cv2
# import os

# # Đường dẫn video
# video_path = r"D:\cdio3\video_den_do.mp4"

# # Tạo thư mục lưu frame
# output_folder = r"D:\cdio3\anhvideo"
# os.makedirs(output_folder, exist_ok=True)

# # Đọc video
# cap = cv2.VideoCapture(video_path)

# frame_count = 0

# while True:
#     ret, frame = cap.read()  # đọc từng frame

#     if not ret:
#         break  # hết video

#     # Lưu frame thành ảnh
#     frame_name = f"{output_folder}/frame_{frame_count:04d}.jpg"
#     cv2.imwrite(frame_name, frame)

#     frame_count += 1

# cap.release()
# print("Done!")